In [12]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 72.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [2]:
import pandas as pd
df = pd.read_csv("../../data/df_processed.csv",index_col=0)
train_mask = df["MathScore"].notna()

y_train = df.loc[train_mask, "MathScore"]
X = df.drop(columns=["MathScore"])
X_train, X_test = X.loc[train_mask], X.loc[~train_mask]

In [3]:
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np

def run_kfold_catboost(X_train, y_train, cat_rows, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        train_pool = Pool(X_tr, y_tr, cat_features=cat_rows)
        val_pool   = Pool(X_val, y_val, cat_features=cat_rows)

        model = CatBoostRegressor(
            depth=8,
            learning_rate=0.05,
            iterations=2000,
            loss_function="RMSE",
            verbose=False
        )

        model.fit(train_pool, eval_set=val_pool)

        preds = model.predict(X_val)
        r2 = r2_score(y_val, preds)
        scores.append(r2)

        print(f"Fold {fold+1}: R² = {r2:.5f}")

    print(f"Mean R²: {np.mean(scores):.5f}")
    return scores

# Usage:
# scores = run_kfold_catboost(X_train, y_train, cat_rows)

In [15]:
cat_rows= ['Year', 'STRATUM', 'CNT', 'CYC']
#run_kfold_catboost(X_train, y_train, cat_rows)

In [4]:
exo_embedding_finetuned_full_train = pd.read_csv("../../data/Embedding_wandb/exo_embedding_finetuned_full_train.csv",index_col=0)
que_embedding_finetuned_full_train = pd.read_csv("../../data/Embedding_wandb/que_embedding_finetuned_full_train.csv",index_col=0)

In [5]:
X_train_j = X_train.join(que_embedding_finetuned_full_train, how="left")
X_train_j = X_train_j.join(exo_embedding_finetuned_full_train, how="left")

In [18]:
cat_rows= ['Year', 'STRATUM', 'CNT', 'CYC']
run_kfold_catboost(X_train_j, y_train, cat_rows)

Fold 1: R² = 0.84838
Fold 2: R² = 0.84815
Fold 3: R² = 0.84658
Fold 4: R² = 0.84686
Fold 5: R² = 0.84782
Mean R²: 0.84756


[0.8483778976695888,
 0.8481506391482703,
 0.846579032457206,
 0.8468617727689252,
 0.8478245654846277]

In [6]:
exo_embedding_dae_full_train = pd.read_csv("../../data/Embedding_wandb/exo_embedding_dae_full_train.csv",index_col=0)
que_embedding_dae_full_train = pd.read_csv("../../data/Embedding_wandb/que_embedding_dae_full_train.csv",index_col=0)
X_train_k = X_train.join(exo_embedding_dae_full_train, how="left")
X_train_k = X_train_k.join(que_embedding_dae_full_train, how="left")

In [23]:
cat_rows= ['Year', 'STRATUM', 'CNT', 'CYC']
run_kfold_catboost(X_train_k, y_train, cat_rows)

Fold 1: R² = 0.84045
Fold 2: R² = 0.84041
Fold 3: R² = 0.83925


KeyboardInterrupt: 

In [7]:
import pandas as pd
from sklearn.linear_model import LinearRegression

def train_and_predict_linreg(X_train, y_train, X_test, filename):
    # modèle linéaire simple
    model = LinearRegression()

    # fit complet
    model.fit(X_train, y_train)

    # prédictions
    preds = model.predict(X_test)

    # submission
    submission = pd.DataFrame(
        {"MathScore": preds},
        index=X_test.index
    )
    submission.index.name = "ID"

    # save
    submission.to_csv(filename)

    return submission


In [10]:
cat_rows= ['Year', 'STRATUM', 'CNT', 'CYC']

In [11]:
X_train_nocat = X_train.drop(columns=cat_rows)
X_test_nocat = X_test.drop(columns=cat_rows)
y = train_and_predict_linreg(X_train_nocat, y_train, X_test_nocat, "test_lin.csv")

In [12]:
default_X_train = pd.read_csv("../../data/X_train.csv", index_col=0)
default_y_train = pd.read_csv("../../data/y_train.csv", index_col=0)

In [13]:
default_X_train.head()

,Year,CNT,CNTRYID,CNTSCHID,CNTSTUID,CYC,NatCen,STRATUM,SUBNATIO,OECD,...,science_q10_total_timing,science_q11_total_timing,science_q12_total_timing,science_q13_total_timing,science_q14_total_timing,science_q15_total_timing,science_q16_total_timing,science_q17_total_timing,science_q18_total_timing,science_q19_total_timing
384002,2022,NLD,528.0,52800132.0,52801144.0,08MS,52800,NLD06,5280000,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1118072,2018,QAZ,31.0,3100106.0,3100424.0,07MS,3100,QAZ0101,310000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
845454,2018,FRA,250.0,25000010.0,25005207.0,07MS,25000,FRA0101,2500000,1.0,...,87686.5,13164.75,1187.199951,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1728613,2015,QES,971.0,97100240.0,97127584.0,06MS,72400,ESP1633,7241600,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1083243,2018,PHL,608.0,60800071.0,60802698.0,07MS,60800,PHL0011,6080000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
default_y_train.head()

,MathScore
384002,116.975422
1118072,73.387560
845454,0.000000
1728613,0.000000
1083243,113.750718


In [15]:
default_X_test = pd.read_csv("../../data/X_train.csv", index_col=0)

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def train_and_predict_linreg(X_train, y_train, X_test, filename):
    model = LinearRegression()

    # fit complet
    model.fit(X_train, y_train)

    # R² sur le train
    train_preds = model.predict(X_train)
    r2 = r2_score(y_train, train_preds)
    print(f"Train R²: {r2:.5f}")

    # prédictions test
    preds = model.predict(X_test)

    # submission
    submission = pd.DataFrame(
        {"MathScore": preds},
        index=X_test.index
    )
    submission.index.name = "ID"

    submission.to_csv(filename)

    return submission


In [17]:
topdrop = default_X_train.isna().sum() + default_X_test.isna().sum()
dxtrainfiltered = default_X_train[topdrop[topdrop == 0].index].select_dtypes("number")
dxtestfiltered = default_X_test[topdrop[topdrop == 0].index].select_dtypes("number")

In [18]:
train_and_predict_linreg(dxtrainfiltered, y_train, dxtestfiltered, "test_lin2.csv")

Train R²: 0.02054


,MathScore
ID,
384002,115.706543
1118072,92.457065
845454,116.419256
1728613,125.738564
1083243,85.885614
...,...
259178,116.978627
1414414,109.645813
131932,85.296699


In [19]:
model = CatBoostRegressor(
    depth=8,
    learning_rate=0.05,
    iterations=2000,
    loss_function="RMSE",
    verbose=False
)

In [ ]:
model.fit(train_pool, eval_set=val_pool)

,Option_CT,Option_FL,Option_ICTQ,Option_WBQ,Option_PQ,Option_TQ,Option_UH,MISSSC,ST004D01T,MATHEASE,...,OCOD2,OCOD3,AGE,GRADE,CNTSTUID,COBN_S,Year,STRATUM,CNT,CYC
384002,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.000000,...,1.897948,-0.617833,0.040208,0.000000,0.343986,0.807771,cat_2022,cat_2,cat_NLD,cat_08MS
1118072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.000000,...,-0.954237,-0.038599,-0.882305,0.000000,-0.016346,-0.184934,cat_2018,cat_0,cat_QAZ,cat_07MS
845454,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.831538,-0.725540,0.330805,0.000000,-0.457253,0.590497,cat_2018,cat_4,cat_FRA,cat_07MS
1728613,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.103625,...,0.127507,0.000000,0.672126,-1.000000,-0.623027,-0.034897,cat_2015,cat_3,cat_QES,cat_06MS
1083243,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,-0.562635,0.668444,-0.369173,-7.179011,-0.152850,-0.815895,cat_2018,cat_2,cat_PHL,cat_07MS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.000000,...,-0.165463,0.305616,0.916215,0.000000,1.307192,0.071915,cat_2022,cat_1,cat_ISR,cat_08MS
1414414,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.103625,...,-0.218462,-0.617833,-0.703583,-1.000000,1.482353,0.816001,cat_2015,cat_1,cat_DEU,cat_06MS
131932,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.000000,...,1.691435,0.668444,0.702439,7.179011,1.067625,0.094002,cat_2022,cat_4,cat_HRV,cat_08MS
671155,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,-0.389542,-0.786094,0.529367,-7.179011,-0.623027,0.361759,cat_2018,cat_1,cat_AUT,cat_07MS


In [23]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
(train_idx, val_idx) = next(kf.split(X_train))

In [27]:
X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

train_pool = Pool(X_tr, y_tr, cat_features=cat_rows)
val_pool   = Pool(X_val, y_val, cat_features=cat_rows)

In [28]:
model.fit(train_pool, eval_set=val_pool)

In [29]:
fi = model.get_feature_importance()
fi

array([4.69492200e-02, 4.12095638e-02, 1.77438040e-01, 2.36619973e-02,
       4.97874386e-02, 8.17557704e-02, 4.76078613e-02, 6.90807535e-02,
       2.74742572e-01, 4.11056769e-01, 5.78987196e-02, 4.23133428e-02,
       8.40458638e-02, 6.69406514e+00, 4.30990010e-01, 6.48116447e-01,
       1.33156490e+00, 1.86537160e-01, 6.45671645e-01, 1.22457741e+00,
       1.12330667e+00, 1.25072941e+00, 3.64029833e-01, 5.99863754e-01,
       5.89271995e+01, 3.57460727e+00, 1.46515614e+00, 5.24505397e-01,
       2.29020513e+00, 1.73113262e+01])

In [30]:
imp_mean = model.get_feature_importance()
feat_imp = pd.DataFrame({
    "feature": X_train.columns,
    "importance": imp_mean
}).sort_values("importance", ascending=False)
feat_imp

,feature,importance
24,CNTSTUID,58.927200
29,CYC,17.311326
13,Attempts,6.694065
25,COBN_S,3.574607
28,CNT,2.290205
26,Year,1.465156
16,IMMIG,1.331565
21,OCOD3,1.250729
19,OCOD1,1.224577
20,OCOD2,1.123307


In [31]:
df

,Option_CT,Option_FL,Option_ICTQ,Option_WBQ,Option_PQ,Option_TQ,Option_UH,MISSSC,ST004D01T,MATHEASE,...,OCOD3,AGE,GRADE,CNTSTUID,COBN_S,Year,STRATUM,CNT,CYC,MathScore
384002,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.000000,...,-0.617833,0.040208,0.000000,0.343986,0.807771,cat_2022,cat_2,cat_NLD,cat_08MS,116.975422
1118072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.000000,...,-0.038599,-0.882305,0.000000,-0.016346,-0.184934,cat_2018,cat_0,cat_QAZ,cat_07MS,73.387560
845454,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,-0.725540,0.330805,0.000000,-0.457253,0.590497,cat_2018,cat_4,cat_FRA,cat_07MS,0.000000
1728613,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.103625,...,0.000000,0.672126,-1.000000,-0.623027,-0.034897,cat_2015,cat_3,cat_QES,cat_06MS,0.000000
1083243,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.668444,-0.369173,-7.179011,-0.152850,-0.815895,cat_2018,cat_2,cat_PHL,cat_07MS,113.750718
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1757496,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.103625,...,0.000000,0.181956,-1.000000,0.203613,-0.034897,cat_2015,cat_3,cat_QAR,cat_06MS,NaN
1414197,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.000000,...,-0.173954,-0.057903,-1.000000,0.203613,0.816001,cat_2015,cat_1,cat_DEU,cat_06MS,NaN
821972,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.305616,-0.429636,0.000000,0.203613,0.527927,cat_2018,cat_3,cat_ESP,cat_07MS,NaN
25376,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,-0.617833,-0.350641,0.000000,0.203613,-0.602652,cat_2022,cat_3,cat_ARG,cat_08MS,NaN
